# E9 (v2 — complete) — Faster R-CNN R50-FPN: retrain (if needed) + detailed per-class AP and precision/recall evaluation

**Context:** notebook 01 does NOT save a Faster R-CNN checkpoint, so this v2 is fully self-contained:
1. **Look for a checkpoint** `frcnn*.pt` in every Add Input — if found, skip training and only evaluate (~20 min).
2. **If none → RETRAIN with notebook 01's exact config**: lr 0.005, SGD momentum 0.9, weight_decay 5e-4,
   StepLR(step=3, gamma=0.1), batch 4, 8 epochs, horizontal-flip augmentation, seed 42 (~6.5–7 h on T4).
   A checkpoint is saved EVERY EPOCH (`frcnn_last.pt` + the epoch number in state.json), so an interrupted session can resume;
   after 8 epochs `frcnn_best.pt` is written TO THE OUTPUT (so retraining is never needed again).
3. **Three independent measurements** on the test split (1,795 images):
   (a) COCO-style per-class AP @50 and @50-95 (torchmetrics, class_metrics=True);
   (b) OPERATING-POINT precision/recall at conf=0.5, IoU=0.5, greedy matching;
   (c) latency in ms/image at batch=1 over 200 images after a 10-image warm-up (reference: 86.3 ms).

**Sanity check:** the recomputed mAP@50 must be ≈ 0.7207 (±1–2 points is acceptable when retraining from scratch — report the NEW number
with the note "retrained checkpoint"; a gap >5 points means the pipeline is wrong and the numbers must NOT be used).

**Output:** `results/e9_frcnn_perclass.csv`, `results/e9_frcnn_operating_point.csv`, `frcnn_best.pt` (checkpoint).

**Add Input:** `ppe-unified-v1` (required). GPU T4 x2 · Internet ON (downloads the COCO pretrained weights). QUICK_TEST=True runs a ~10 min pipeline check.

In [ ]:
KEY_PREFIX = "e9_"
# ==================== CONFIG ====================
QUICK_TEST = False    # True = 1 epoch on 3% of the data, pipeline check only; quick numbers must NOT go into the report
SEED = 42
EPOCHS      = 1 if QUICK_TEST else 8
LR, MOMENTUM, WD = 0.005, 0.9, 5e-4
BATCH       = 4

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) for k in _s): STATE.update(_s); print("Resumed state:", list(_s))
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q torchmetrics
import torch, torchvision, numpy as np
assert torch.cuda.is_available(), "Enable GPU T4 x2!"
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| torchvision:", torchvision.__version__)


In [ ]:
from pathlib import Path
import yaml

def find_dataset_root():
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Dataset ppe-unified-v1 has not been Added as Input!")

DS_ROOT = find_dataset_root()
NAMES = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
if isinstance(NAMES, dict): NAMES = [NAMES[k] for k in sorted(NAMES)]
NC = len(NAMES) + 1   # +1 for background (torchvision convention: 0 = background)
print("Dataset:", DS_ROOT, "| 12 classes:", NAMES)


In [ ]:
# ---------- PyTorch dataset built from the YOLO labels (shared by train/test) ----------
from PIL import Image
from torchvision.transforms import functional as TF

class PPEDataset(torch.utils.data.Dataset):
    """Read a PPE-Unified split; normalised YOLO labels -> pixel xyxy boxes; class +1 (0 = background)."""
    def __init__(self, split, train_aug=False, fraction=1.0):
        self.files = sorted(glob.glob(f"{DS_ROOT}/{split}/labels/*.txt"))
        if fraction < 1.0: self.files = self.files[: max(8, int(len(self.files) * fraction))]
        self.train_aug = train_aug
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        lf = self.files[i]
        img_path = lf.replace("/labels/", "/images/").rsplit(".", 1)[0] + ".jpg"
        img = Image.open(img_path).convert("RGB"); W, H = img.size
        boxes, labels = [], []
        for line in open(lf):
            p = line.split()
            if len(p) < 5: continue
            c, xc, yc, w, h = int(p[0]), *map(float, p[1:5])
            x1, y1, x2, y2 = (xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H
            if x2 - x1 < 2 or y2 - y1 < 2: continue
            boxes.append([x1, y1, x2, y2]); labels.append(c + 1)
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros(0, dtype=torch.int64)
        x = TF.to_tensor(img)
        if self.train_aug and random.random() < 0.5:      # horizontal flip (the only augmentation — as in notebook 01)
            x = torch.flip(x, [2])
            if len(boxes): boxes = torch.stack([W - boxes[:, 2], boxes[:, 1], W - boxes[:, 0], boxes[:, 3]], 1)
        return x, {"boxes": boxes, "labels": labels}

collate = lambda b: tuple(zip(*b))

# ---------- Model + checkpoint lookup ----------
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def new_model(pretrained):
    m = fasterrcnn_resnet50_fpn(weights="DEFAULT" if pretrained else None)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NC)
    return m

ckpt = None
for pat in ("frcnn*best*.pt", "frcnn*last*.pt", "frcnn*.pth", "fasterrcnn*.pt"):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))
    if hits: ckpt = hits[0]; break
print("Checkpoint found:", ckpt or "NONE — will retrain from the COCO pretrained weights")


In [ ]:
# ---------- TRAIN (skipped entirely if a checkpoint exists) ----------
model = new_model(pretrained=(ckpt is None)).to(DEV)
if ckpt:
    model.load_state_dict(torch.load(ckpt, map_location=DEV)); print("Checkpoint loaded — skipping training.")
elif done("e9_train"):
    model.load_state_dict(torch.load(f"{WORK}/frcnn_best.pt", map_location=DEV)); print("Training already completed in a previous session.")
else:
    train_ds = PPEDataset("train", train_aug=True, fraction=0.03 if QUICK_TEST else 1.0)
    loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                                         num_workers=2, collate_fn=collate, pin_memory=True)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=MOMENTUM, weight_decay=WD)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.1)
    start_ep = int(STATE.get("e9_epoch", 0))
    # mid-run resume: load the last checkpoint from a previous session's Added output (if any)
    prev_last = sorted(glob.glob("/kaggle/input/**/frcnn_last.pt", recursive=True))
    if start_ep > 0 and prev_last:
        model.load_state_dict(torch.load(prev_last[0], map_location=DEV))
        for _ in range(start_ep): sched.step()
        print(f"Resuming training from epoch {start_ep + 1}/{EPOCHS}")
    model.train(); t0 = time.time()
    for ep in range(start_ep, EPOCHS):
        run_loss, nb = 0.0, 0
        for imgs, targets in loader:
            imgs = [i.to(DEV) for i in imgs]
            targets = [{k: v.to(DEV) for k, v in t.items()} for t in targets]
            losses = model(imgs, targets)
            loss = sum(losses.values())
            opt.zero_grad(); loss.backward(); opt.step()
            run_loss += float(loss); nb += 1
            if QUICK_TEST and nb >= 30: break
        sched.step()
        torch.save(model.state_dict(), f"{WORK}/frcnn_last.pt")          # checkpoint EVERY epoch
        STATE["e9_epoch"] = ep + 1; save_state()
        print(f"epoch {ep+1}/{EPOCHS} | mean loss {run_loss/max(nb,1):.4f} | {round((time.time()-t0)/60,1)} min")
    torch.save(model.state_dict(), f"{WORK}/frcnn_best.pt")
    STATE["e9_train"] = {"epochs": EPOCHS, "min": round((time.time() - t0) / 60, 1)}; save_state()
    print("Training complete — frcnn_best.pt written to the output.")
model.eval();


In [ ]:
# ---------- EVALUATION: (1) COCO per-class AP  (2) operating-point P/R  (3) latency ----------
from torchmetrics.detection import MeanAveragePrecision
import pandas as pd

test_ds = PPEDataset("test", fraction=0.05 if QUICK_TEST else 1.0)
print("Test images:", len(test_ds))
metric = MeanAveragePrecision(class_metrics=True, iou_type="bbox")
tp = {c: 0 for c in range(len(NAMES))}; fp = {c: 0 for c in range(len(NAMES))}; fn = {c: 0 for c in range(len(NAMES))}
CONF, IOU_T = 0.5, 0.5

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0

lat = []
with torch.no_grad():
    for i in range(len(test_ds)):
        x, gt = test_ds[i]
        t0 = time.time(); out = model([x.to(DEV)])[0]; torch.cuda.synchronize()
        if 10 <= i < 210: lat.append((time.time() - t0) * 1000)
        metric.update([{"boxes": out["boxes"].cpu(), "labels": out["labels"].cpu() - 1, "scores": out["scores"].cpu()}],
                      [{"boxes": gt["boxes"], "labels": gt["labels"] - 1}])
        pb = out["boxes"].cpu().numpy(); pl = (out["labels"].cpu().numpy() - 1); ps = out["scores"].cpu().numpy()
        gtb = gt["boxes"].numpy(); gtl = (gt["labels"].numpy() - 1)
        keep = ps >= CONF; pb, pl, ps_k = pb[keep], pl[keep], ps[keep]
        matched = set()
        for j in np.argsort(-ps_k):
            best, bi = 0, -1
            for k in range(len(gtb)):
                if k in matched or gtl[k] != pl[j]: continue
                v = iou(pb[j], gtb[k])
                if v > best: best, bi = v, k
            if best >= IOU_T: tp[int(pl[j])] += 1; matched.add(bi)
            else: fp[int(pl[j])] += 1
        for k in range(len(gtb)):
            if k not in matched: fn[int(gtl[k])] += 1

res = metric.compute()
per_cls_50   = res.get("map_50_per_class", None)
per_cls_5095 = res.get("map_per_class", None)
rows = []
for i, n in enumerate(NAMES):
    p = tp[i] / max(tp[i] + fp[i], 1); r = tp[i] / max(tp[i] + fn[i], 1)
    ap5095 = float(per_cls_5095[i]) if per_cls_5095 is not None and per_cls_5095.numel() > i else None
    ap50   = float(per_cls_50[i])   if per_cls_50 is not None and hasattr(per_cls_50, "numel") and per_cls_50.numel() > i else None
    rows.append({"class": n, "AP50": round(ap50, 4) if ap50 is not None else None,
                 "AP50_95": round(ap5095, 4) if ap5095 is not None else None,
                 "precision@0.5": round(p, 4), "recall@0.5": round(r, 4),
                 "TP": tp[i], "FP": fp[i], "FN": fn[i]})
df = pd.DataFrame(rows); df.to_csv(f"{WORK}/results/e9_frcnn_perclass.csv", index=False)
P = sum(tp.values()) / max(sum(tp.values()) + sum(fp.values()), 1)
R = sum(tp.values()) / max(sum(tp.values()) + sum(fn.values()), 1)
op = {"mAP50": round(float(res["map_50"]), 4), "mAP50_95": round(float(res["map"]), 4),
      "precision@0.5": round(P, 4), "recall@0.5": round(R, 4),
      "infer_ms_bs1": round(float(np.mean(lat)), 2) if lat else None,
      "checkpoint": "input" if ckpt else "retrained"}
pd.DataFrame([op]).to_csv(f"{WORK}/results/e9_frcnn_operating_point.csv", index=False)
STATE["e9_done"] = op; save_state()
print(df.to_string(index=False)); print(op)
print("SANITY: mAP50 ~0.7207 (±1-2 points if retrained). A gap >5 points means the pipeline is wrong — do NOT use the numbers.")


In [ ]:
# ==================== CLEAN UP OUTPUT ====================
# KEEP: results/, state.json, frcnn_best.pt, frcnn_last.pt (permanently reusable). Delete the rest.
KEEP = {"results", "state.json", "frcnn_best.pt", "frcnn_last.pt"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Final output:", sorted(os.listdir(WORK)))
